In [0]:
# --------------------------------------------------------------------------------------------------------
# SCRIPT: 5.1a_cpis_timeline_silver
# LOCAL: 2_silver/src/5_auditoria_cpis/
# OBJETIVO: Tratamento de datas e cálculo de duração das CPIs. Fornece fonte para o script 
#           3_gold/../5_auditoria_cpis/5.1_visao_consolidada_cpis e tambem 5.4_rede_convocados_cpis
# ENTREGA: Tabela específica de CPIs com timeline e membros por fase
# --------------------------------------------------------------------------------------------------------

from pyspark.sql.functions import col, datediff, to_date, when, lit

# Ler a tabela Bronze gerada pelo Mock
df_cpis = spark.table("bronze_cpis_lista")

# Transformação de Datas e Cálculo de Timeline
df_silver_cpis = df_cpis.select(
    col("id").alias("id_cpi"),
    col("sigla"),
    col("ementa").alias("descricao"),
    to_date(col("dataApresentacao")).alias("data_inicio"),
    to_date(col("dataFim")).alias("data_fim"),
    col("status_cpi")
).withColumn("duracao_atual_dias", datediff(col("data_fim"), col("data_inicio"))) \
 .withColumn("status_prazo", 
             when(col("duracao_atual_dias") > 120, lit("Excedeu Prazo Regimental"))
             .when(col("data_fim").isNull(), lit("Em Aberto"))
             .otherwise(lit("Dentro do Prazo")))

# Salvar na Silver
df_silver_cpis.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_cpis_detalhes")

print("Tabela silver_cpis_detalhes criada com sucesso!")
display(df_silver_cpis)